In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [6]:
# from huggingface_hub import snapshot_download
# import time

# while True:
#     try:
#         snapshot_download(
#             repo_id="nucklehead/ht-voice-dataset", 
#             repo_type="dataset", local_dir="./ht-voice-dataset")
#         break
#     except Exception as e:
#         print(e)
#     time.sleep(60)

In [ ]:
files= glob('ht-voice-dataset/clips/*.csv')

['ht-voice-dataset/clips/other.csv',
 'ht-voice-dataset/clips/dev.csv',
 'ht-voice-dataset/clips/train-all.csv',
 'ht-voice-dataset/clips/validated.csv',
 'ht-voice-dataset/clips/test.csv',
 'ht-voice-dataset/clips/train.csv']

In [12]:
rows = pd.read_csv('ht-voice-dataset/clips/train.csv').to_dict(orient = 'records')
rows[0]

{'wav_filename': 'EYMW_2Q09.wav',
 'wav_filesize': 173714,
 'transcript': 'anpil nan yo pat fè swivan sa lwa sou bidjè a di pou yo fè'}

In [26]:
base = 'ht-voice-dataset_audio'
!mkdir {base}

In [34]:
def loop(rows):
    rows, _ = rows
    data = []
    for row in tqdm(rows):
        f = os.path.join('ht-voice-dataset/clips', row['wav_filename'])
        if not os.path.exists(f):
            continue

        t = row['transcript'].strip()
        if len(t) < 2:
            continue
        
        audio_filename = f.replace('/', '-').replace('.wav', '.mp3')
        audio_filename = os.path.join(base, audio_filename)

        audio_np, sr = sf.read(f)
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)

        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': f"{base}"
        })
    return data

In [35]:
data = loop((rows[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 17.57it/s]


In [36]:
data = multiprocessing(rows, loop, cores = 20)

  2%|▏         | 2/87 [00:00<00:06, 13.47it/s]

100%|██████████| 87/87 [00:06<00:00, 12.66it/s]


In [37]:
len(data)

1743

In [38]:
audio_files = [d['audio_filename'] for d in data]

with open('ht-voice-dataset-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [40]:
# !zip -rq ht-voice-dataset_audio.zip ht-voice-dataset_audio
# !hf upload malaysia-ai/Multilingual-TTS ht-voice-dataset_audio.zip --repo-type=dataset